In [57]:
import pandas as pd
import numpy as np
import spacy
from spacy import displacy
from spacy.matcher import PhraseMatcher
import json
import re

In [22]:
with open("scraping/scrape_keywords/medicine_flattened.json") as file:
    keywords = json.load(file)

# Aspect Based Sentiment Analysis

spaCy
- English model: [en_core_web_sm](https://spacy.io/models/en)
- Chinese model: [zh_core_web_sm](https://spacy.io/models/zh)

In [61]:
def nlp_factory(keywords, lang = "en"):
    if lang == "en":
        nlp = spacy.load("en_core_web_sm")
        phrases = keywords['en']
        matcher = PhraseMatcher(nlp.vocab, attr="LEMMA") # lemmatize
        patterns = [nlp(text) for text in phrases]
        matcher.add("MEDICINES", patterns)
    else:
        nlp = spacy.load("zh_core_web_sm")
        # cannot match Chinese elegantly
        matcher = None
    return {"nlp": nlp, "matcher": matcher}

In [78]:
def merge_ranges(ranges):
    if len(ranges)==0:
        return ranges
    sorted_ranges = sorted(ranges, key=lambda x: x[0])
    merged = [list(sorted_ranges[0])]
    for start, end in sorted_ranges[1:]:
        if start <= merged[-1][1]:
            merged[-1][1] = max(merged[-1][1], end)
        else:
            merged.append(list(start, end))
    return merged

In [88]:
class custom_nlp():
    def __init__(self, keywords, lang='en'):
        self.keywords = keywords
        self.lang = lang
        nlp_group = nlp_factory(keywords, lang)
        self.nlp = nlp_group['nlp']
        self.matcher = nlp_group['matcher']
    
    def __call__(self, text):
        doc = self.nlp(text)
        # english
        if self.lang == 'en':
            with doc.retokenize() as retokenizer:
                for match_id, start, end in self.matcher(doc):
                    span = doc[start:end]
                    retokenizer.merge(span)
            return doc
        # chinese
        ranges = []
        for phrase in self.keywords['cn']:
            for match in re.finditer(re.escape(phrase), doc.text):
                start_char, end_char = match.span()
                ranges.append((start_char, end_char))
        merged_ranges = merge_ranges(ranges)
        with doc.retokenize() as retokenizer:
            for start, end in merged_ranges:
                span = doc.char_span(start, end, alignment_mode="expand") # align to tokens that cover the whole sequence
                if span is not None and len(span) > 1:
                    retokenizer.merge(span)
        return doc


In [89]:
nlp_dict = {
    "en": custom_nlp(keywords, "en"),
    "cn": custom_nlp(keywords, "cn")
}

In [70]:
displacy.render(nlp_dict['en']("Lianhua Qingwen Jiaonang was so bitter"))

In [71]:
displacy.render(nlp_dict['en']("I forgot to put on my steroid creams before I left home for school this morning"))

In [72]:
displacy.render(nlp_dict['cn']("二型糖尿病是由于胰岛素抵抗导致的"))

In [90]:
displacy.render(nlp_dict['cn']("连花清瘟胶囊是治疗新冠的良药"))

In [91]:
%%timeit
nlp_dict['en']("Lianhua Qingwen Jiaonang was so bitter")

2.41 ms ± 157 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [92]:
%%timeit
nlp_dict['cn']("连花清瘟胶囊是治疗新冠的良药")

3.04 ms ± 229 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [ ]:
def prune_sentence_for_aspect(text, aspect, nlp):
    doc = nlp(text)
    target_token = None
    for token in doc:
        if token.text.lower() == aspect.lower():
            target_token = token
            break
            
    # If the aspect isn't in the sentence, return None
    if not target_token:
        return None

    relevant_tokens = set()

    # 2. Add the target itself and its immediate modifiers (e.g., "the strong antibiotics")
    for token in target_token.subtree:
        relevant_tokens.add(token)

    # 3. Find the grammatical head (usually the main verb governing the aspect)
    head = target_token.head
    relevant_tokens.add(head)

    # 4. Traverse the children of the head to capture sentiment payloads
    # We specifically target adjectival complements, adverbs, negations, and objects.
    valid_dependencies = {
        'acomp',   # Adjectival complement (e.g., "is effective")
        'advmod',  # Adverbial modifier (e.g., "highly", "completely")
        'neg',     # Negation (e.g., "not", "never")
        'dobj',    # Direct object (e.g., "cleared the infection")
        'xcomp',   # Open clausal complement (e.g., "seems to work")
        'attr'     # Attribute (e.g., "is a scam")
    }

    for child in head.children:
        if child.dep_ in valid_dependencies:
            # If a child is relevant, capture its entire subtree
            # (e.g., if child is "infection", capture "my severe infection")
            for sub_token in child.subtree:
                relevant_tokens.add(sub_token)

    # 5. Sort the collected tokens by their original index to preserve syntax
    sorted_tokens = sorted(list(relevant_tokens), key=lambda t: t.i)
    
    # 6. Reconstruct the string
    pruned_text = " ".join([token.text for token in sorted_tokens])
    
    return pruned_text

# ==========================================
# Testing the Pipeline
# ==========================================

# Test Case 1: Complex compound sentence with unrelated negative sentiment
text_1 = "The hospital waiting room was an absolute nightmare, but the antibiotics completely cleared my severe infection."
aspect_1 = "antibiotics"

# Test Case 2: Aspect modified by a negation and adjectival complement
text_2 = "To be honest, the acupuncture was not very effective for my back."
aspect_2 = "acupuncture"

print(f"Original 1: {text_1}")
print(f"Pruned 1:   {prune_sentence_for_aspect(text_1, aspect_1)}\n")

print(f"Original 2: {text_2}")
print(f"Pruned 2:   {prune_sentence_for_aspect(text_2, aspect_2)}")